# M1 — Semantic Scholar references (Colab + Drive)

Chạy **sau** notebook crawl arXiv của người 1 (`M1_Colab_Concurrent_Limited.ipynb`).
Quy trình: GitHub branch → Drive → đọc ID từ checkpoint `batch_*.tar.gz` → crawl references (resume) → xuất 2 bảng `papers` / `references` + validate.

- **1 request / publication** (S2 không phân version). Có `S2_API_KEY` thì ổn định hơn: thêm **Secret** tên `S2_API_KEY` trong Colab (biểu tượng chìa khóa). **Không dán key vào notebook.**
- Mỗi paper có trạng thái rõ ràng: `ok` / `not_indexed` (S2 chưa ghi nhận, 404) / `failed` (429, 5xx, mất mạng). Bài thật sự **0 reference** vẫn được ghi file nên không bị gọi lại.
- Resume: `ok` và `not_indexed` bị bỏ qua; `failed` được thử lại ở lần chạy sau. Muốn thử lại cả `not_indexed` (S2 index chậm 1–7 ngày) → `RETRY_NOT_INDEXED = True`.
- Lưu **toàn bộ** reference (kể cả không có arXiv ID); bảng `references` chỉ giữ ref có arXiv ID.
- Lỗi từng paper không dừng cả run.

**Cần push code Python mới nhất lên nhánh của nhóm trước khi chạy notebook.**

## SET UP — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuandatedu/arXir_project.git"  # TODO: thay URL repo nhóm
BRANCH = "M1-new"                    # TODO: tên branch thật
PROJECT = Path("/content/project")
assert "OWNER" not in REPO_URL, "Bạn chưa thay URL GitHub"

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_m1_pipeline.py"], check=True)
print("Repo:", PROJECT, "| Branch:", BRANCH)

## SET UP — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Trỏ đúng thư mục Drive của range

`RANGE_ID` và đường dẫn `DATA_DIR` phải **trùng với notebook crawl của người 1** để tìm được checkpoint. Dữ liệu references được ghi xen vào cạnh đó:

```text
DATA_DIR/
  batch_00000.tar.gz + .json     # checkpoint của người 1 (nguyên trạng)
  references/<folder>/references_s2.json   # MỌI ref đã crawl (resume tại đây)
  tables/papers.json, references.json, metrics_s2_references.json, s2_crawl_records.csv
```

In [ ]:
RANGE_ID = 4

In [ ]:

import os
from pathlib import Path

# 1. DATA_DIR: checkpoint của Người 1 (chỉ đọc)
DATA_DIR = Path(
    f"/content/drive/MyDrive/arxiv-project-data/m1/range_{RANGE_ID}_dev"
)

# 2. Thư mục riêng trên Drive để lưu kết quả S2
MY_REFS_DIR = Path(
    f"/content/drive/MyDrive/references/range_{RANGE_ID}"
)

# reference/: lưu JSON riêng cho từng paper
# tables/: lưu papers, references và metrics
REFS_ROOT = MY_REFS_DIR / "reference"
TABLES_DIR = MY_REFS_DIR / "tables"

for path in (MY_REFS_DIR, REFS_ROOT, TABLES_DIR):
    path.mkdir(parents=True, exist_ok=True)

# 3. Kiểm tra checkpoint của Người 1
manifests = sorted(DATA_DIR.glob("batch_*.json"))

assert manifests, (
    f"Chưa tìm thấy batch_*.json trong {DATA_DIR} "
    "— kiểm tra lại lối tắt Drive"
)

print(f"Range: {RANGE_ID}")
print(f"Đọc checkpoint từ: {DATA_DIR}")
print(f"Số batch tìm thấy: {len(manifests)}")
print(f"Lưu references tại: {REFS_ROOT}")
print(f"Lưu tables tại: {TABLES_DIR}")

# 4. API key lấy từ Colab Secrets
try:
    from google.colab import userdata

    api_key = userdata.get("S2_API_KEY")

    if not api_key:
        raise ValueError("S2_API_KEY không tồn tại hoặc đang trống.")

    os.environ["S2_API_KEY"] = api_key
    print("Đã nạp S2_API_KEY từ Colab Secrets.")

except Exception as e:
    os.environ.pop("S2_API_KEY", None)
    print(f"Không nạp được S2_API_KEY: {e}")
    print("Sẽ chạy không key (chậm hơn, dễ bị 429).")

# 5. Cấu hình đường dẫn cho pipeline và metrics
from src.common import config
import src.common.metrics as metrics_module

config.WORKING_DIR = MY_REFS_DIR
config.TABLES_DIR = TABLES_DIR
metrics_module.TABLES_DIR = TABLES_DIR

# False: không tự động thử lại các paper từng được xác định là chưa index
RETRY_NOT_INDEXED = False

print("\nTABLES_DIR =", config.TABLES_DIR)
print("REFS_ROOT =", REFS_ROOT)


## 2 — Đọc danh sách paper từ checkpoint `batch_*.tar.gz`

Mỗi archive chứa `papers.jsonl` (metadata người 1 đã crawl). Lấy metadata của **mọi** bài, không lọc theo trạng thái tải TeX: bài chỉ có PDF vẫn là một hàng của bảng `papers` và vẫn có references. Bài S2/arXiv không tìm thấy (`not_found`) sẽ chỉ được báo cáo, không làm dừng notebook.

In [ ]:
import json
import tarfile

from src.common.ids import normalize_arxiv_id
from src.m1_crawling.assigned_ranges import assigned_ids

OUTPUT_TABLES_DIR = TABLES_DIR
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)


def _read_jsonl_from_archive(archive_path: Path, filename: str) -> list[dict]:
    """Đọc một file jsonl trong archive .tar.gz, không giải nén ra đĩa."""
    records = []
    with tarfile.open(archive_path, "r:gz") as tar:
        try:
            member = tar.extractfile(filename)
        except KeyError:
            raise ValueError(f"{archive_path.name} thiếu {filename}")
        if member:
            for line in member:
                line = line.strip()
                if line:
                    records.append(json.loads(line.decode("utf-8")))
    return records


def load_all_papers_from_all_batches() -> list[dict]:
    papers_dict = {}
    manifests = sorted(DATA_DIR.glob("batch_*.json"))
    print(f"Range {RANGE_ID} | Tìm thấy {len(manifests)} batch trong {DATA_DIR}")
    for index, manifest in enumerate(manifests, start=1):
        archive = manifest.with_suffix(".tar.gz")
        if not archive.exists():
            print(f"[warn] Bỏ qua (thiếu .tar.gz): {archive.name}")
            continue
        try:
            rows = _read_jsonl_from_archive(archive, "papers.jsonl")
        except (tarfile.TarError, EOFError, OSError, ValueError) as exc:
            print(f"[warn] Batch lỗi, cần crawl lại: {archive.name} ({exc})")
            continue
        for row in rows:
            try:
                paper_id = normalize_arxiv_id(row.get("arxiv_id") or "")
            except ValueError:
                print(f"[warn] Bỏ qua dòng có ID không hợp lệ: {row.get('arxiv_id')!r}")
                continue
            papers_dict[paper_id] = {
                "arxiv_id": paper_id,
                "title": row.get("title", ""),
                "authors": row.get("authors", []),
                "submission_date": row.get("submission_date", ""),
                "revised_dates": row.get("revised_dates", []),
                "venue": row.get("venue", None),
            }
        if index % 10 == 0 or index == len(manifests):
            print(f"  -> {index}/{len(manifests)} batch (tích lũy {len(papers_dict)} bài)")
    return sorted(papers_dict.values(), key=lambda p: p["arxiv_id"])


papers_meta = load_all_papers_from_all_batches()
print(f"\nHOÀN THÀNH RANGE {RANGE_ID}: {len(papers_meta)} bài báo có metadata.")

# Đối chiếu với danh sách ID đề giao (chỉ báo cáo, không assert)
expected = set(assigned_ids(RANGE_ID))
got = {p["arxiv_id"] for p in papers_meta}
missing = sorted(expected - got)
print(f"Đề giao: {len(expected)} | Có metadata: {len(got & expected)} | "
      f"Thiếu: {len(missing)} | Thừa ngoài range: {len(got - expected)}")
if missing:
    print("[warn] Chưa có metadata (batch chưa crawl, lỗi, hoặc ID không tồn tại trên arXiv):",
          missing[:10], "..." if len(missing) > 10 else "")
    print("       Xem reports/crawl_records.csv của người 1 để biết status (error / not_found).")


## 3 — Crawl references (resume, có đo runtime)

Mỗi paper được lưu vào `REFS_ROOT/<folder>/`: `references_s2.json` (mọi reference, mỗi dòng một JSON) và `s2_status.json` (trạng thái, số reference, thời gian, lỗi). Ghi nguyên tử (ghi tạm rồi đổi tên), nên session Colab bị ngắt giữa chừng cũng không để lại file dở. Chạy lại cell này để tiếp tục.

In [ ]:
import logging
import time
from collections import Counter

from src.common.io import write_json
from src.common.metrics import track_run
from src.m1_crawling.semantic_scholar import fetch_s2_references

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

if "papers_meta" not in globals() or not papers_meta:
    raise RuntimeError("'papers_meta' chưa được tạo! Hãy chạy lại cell ở Mục 2 trước.")


# Bắt đầu đo thời gian toàn bộ cell
started_at = time.perf_counter()

with track_run("s2_references") as metrics:
    metrics["range_id"] = RANGE_ID
    metrics["n_ids"] = len(papers_meta)

    # Crawl references từ Semantic Scholar
    results = fetch_s2_references(
        papers_meta,
        REFS_ROOT,
        api_key=os.environ.get("S2_API_KEY"),
        retry_not_indexed=RETRY_NOT_INDEXED,
        progress_every=50,
    )

    # Phân loại trạng thái
    counts = Counter(r["status"] for r in results)

    ok_rows = [r for r in results if r["status"] == "ok"]
    not_indexed_rows = [
        r for r in results if r["status"] == "not_indexed"
    ]
    failed_rows = [
        r for r in results if r["status"] == "failed"
    ]

    # Phân biệt cache và dữ liệu được xử lý trong lần chạy này
    cached = [r for r in results if r.get("from_cache", False)]
    fresh = [r for r in results if not r.get("from_cache", False)]

    # Chỉ tính references từ kết quả hợp lệ
    n_refs_total = sum(
        r.get("n_refs", 0) for r in ok_rows
    )
    n_refs_with_arxiv = sum(
        r.get("n_arxiv", 0) for r in ok_rows
    )
    n_refs_without_arxiv = max(
        0, n_refs_total - n_refs_with_arxiv
    )

    # Số paper thành công nhưng không có references
    n_zero_refs = sum(
        1 for r in ok_rows if r.get("n_refs", 0) == 0
    )

    # Thời gian xử lý các paper không lấy từ cache
    fetched_times = [
        float(r.get("elapsed_s", 0) or 0)
        for r in fresh
    ]

    total_fetch_elapsed = sum(fetched_times)

    avg_fetch_time = (
        total_fetch_elapsed / len(fetched_times)
        if fetched_times else 0.0
    )

    n_total = len(results)
    n_fetched = len(fresh)
    n_cached = len(cached)

    # Thời gian chạy thực tế của toàn bộ cell
    cell_elapsed = time.perf_counter() - started_at

    # Metrics S2
    metrics["s2_stats"] = {
        # Số lượng paper
        "n_papers": n_total,
        "n_ok": len(ok_rows),
        "n_not_indexed": len(not_indexed_rows),
        "n_failed": len(failed_rows),
        "n_cached": n_cached,
        "n_fetched_this_run": n_fetched,

        # Tỷ lệ
        "success_rate": (
            len(ok_rows) / n_total if n_total else 0.0
        ),
        "failure_rate": (
            len(failed_rows) / n_total if n_total else 0.0
        ),
        "not_indexed_rate": (
            len(not_indexed_rows) / n_total if n_total else 0.0
        ),
        "cache_hit_rate": (
            n_cached / n_total if n_total else 0.0
        ),

        # References
        "n_refs_total": n_refs_total,
        "n_refs_with_arxiv": n_refs_with_arxiv,
        "n_refs_without_arxiv": n_refs_without_arxiv,
        "n_zero_refs": n_zero_refs,
        "avg_refs_per_ok_paper": (
            n_refs_total / len(ok_rows) if ok_rows else 0.0
        ),

        # Hiệu năng
        "avg_s2_seconds_per_fetched_paper": avg_fetch_time,
        "total_fetch_elapsed_sec": total_fetch_elapsed,
        "cell_elapsed_sec": cell_elapsed,
        "papers_per_second": (
            n_total / cell_elapsed if cell_elapsed > 0 else 0.0
        ),
        "status_counts": dict(counts),
    }

# Ghi metrics lên Google Drive
write_json(TABLES_DIR / "metrics_s2_references.json", metrics)

stats = metrics["s2_stats"]

print(f"\nKẾT QUẢ S2 CRAWLING — RANGE {RANGE_ID}")
print("=" * 55)

print("\nPAPERS")
print(f"Total papers:       {stats['n_papers']}")
print(f"Successful:         {stats['n_ok']}")
print(f"Not indexed:        {stats['n_not_indexed']}")
print(f"Failed:             {stats['n_failed']}")
print(f"Cached:             {stats['n_cached']}")
print(f"Fetched this run:   {stats['n_fetched_this_run']}")

print("\nRATES")
print(f"Success rate:       {stats['success_rate']:.2%}")
print(f"Failure rate:       {stats['failure_rate']:.2%}")
print(f"Not indexed rate:   {stats['not_indexed_rate']:.2%}")
print(f"Cache hit rate:     {stats['cache_hit_rate']:.2%}")

print("\nREFERENCES")
print(f"Total references:   {stats['n_refs_total']}")
print(f"With arXiv ID:      {stats['n_refs_with_arxiv']}")
print(f"Without arXiv ID:   {stats['n_refs_without_arxiv']}")
print(f"Papers with 0 refs: {stats['n_zero_refs']}")
print(f"Avg refs / OK paper:{stats['avg_refs_per_ok_paper']:.2f}")

print("\nPERFORMANCE")
print(
    f"Avg fetch time:     "
    f"{stats['avg_s2_seconds_per_fetched_paper']:.3f} sec/paper"
)
print(
    f"Total fetch time:   "
    f"{stats['total_fetch_elapsed_sec']:.2f} sec"
)
print(f"Cell elapsed:       {stats['cell_elapsed_sec']:.2f} sec")

print("\nSYSTEM RESOURCES")
print(
    f"Wall-clock:         "
    f"{metrics.get('duration_sec', 0):.2f} sec"
)
print(
    f"Peak RAM:           "
    f"{metrics.get('peak_rss_bytes', 0) / (1024 ** 2):.2f} MiB"
)
print(
    f"Average RAM:        "
    f"{metrics.get('avg_rss_bytes', 0) / (1024 ** 2):.2f} MiB"
)

# Hiển thị một số paper thất bại để kiểm tra
failed = failed_rows

if failed:
    print(f"\n[WARN] {len(failed)} papers failed.")
    for r in failed[:10]:
        print(
            " ",
            r.get("arxiv_id"),
            ":",
            r.get("error", "Unknown error"),
        )

print("\nStatus breakdown:")
for status, count in sorted(counts.items()):
    print(f"  {status}: {count}")



In [ ]:
# Ghi log từng paper (CSV) để đối chiếu / gửi người xử lý dữ liệu.
# `results` gồm mọi paper (kể cả lấy từ lần chạy trước), nên file này luôn đầy đủ.
import csv

records_csv = TABLES_DIR / "s2_crawl_records.csv"
fields = ["arxiv_id", "status", "n_refs", "n_arxiv", "from_cache", "elapsed_s", "error"]
with records_csv.open("w", newline="", encoding="utf-8-sig") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    for record in results:
        writer.writerow({key: record.get(key, "") for key in fields})
print("Đã ghi", records_csv)


## 4 — Recovery: chạy lại phần references dở dang

Chỉ chạy cell crawl ở mục 3 thêm lần nữa là đủ: paper đã lưu sẽ `skipped`, paper lỗi/chưa tới sẽ được thử lại.
Nếu 429 dồn dập thì chờ ≥ 5 phút rồi chạy lại; **không tăng tốc hoặc mở song song**.

## 5 — Xuất 2 bảng `papers` / `references` + validate

- `papers`: metadata đọc từ checkpoint (1 hàng / publication).
- `references`: chỉ ref **có arXiv ID** (matching pool của M3); mọi ref vẫn nằm trong `references/<folder>/references_s2.json`.
Kết quả validate (thiếu thông tin / sai kiểu / trùng lặp) được in ra và ghi vào metrics.

In [ ]:
from src.common.io import read_jsonl
from src.common.schemas import PaperRow, ReferenceRow
from src.m1_crawling.semantic_scholar import REFS_FILENAME
from src.m1_crawling.tables import (
    dedupe_reference_rows,
    s2_records_to_reference_rows,
    validate_m1_tables,
    write_m1_tables,
)
from src.common.ids import folder_name_for_arxiv_id

# Khóa chắc chắn đường dẫn xuất file về Drive cá nhân
config.WORKING_DIR = MY_REFS_DIR
config.TABLES_DIR = TABLES_DIR

paper_rows: list[PaperRow] = []
reference_rows: list[ReferenceRow] = []
for row in papers_meta:                      # papers_meta đã unique theo arxiv_id
    base = row["arxiv_id"]
    paper_rows.append(PaperRow(
        arxiv_id=base,
        title=row.get("title", ""),
        authors=row.get("authors") or [],
        submission_date=row.get("submission_date", ""),
        revised_dates=row.get("revised_dates") or [],
        venue=row.get("venue"),
    ))
    target = REFS_ROOT / folder_name_for_arxiv_id(base) / REFS_FILENAME
    if target.exists():
        reference_rows.extend(s2_records_to_reference_rows(base, read_jsonl(target)))

n_before = len(reference_rows)
reference_rows = dedupe_reference_rows(reference_rows)
print(f"references: {n_before} -> {len(reference_rows)} sau khi bỏ trùng cặp (arxiv_id, ref_arxiv_id)")

report = validate_m1_tables(paper_rows, reference_rows)
print("validate ok =", report["ok"], "| papers =", report["n_papers"],
      "| references =", report["n_references"],
      "| errors =", report["n_errors"], "| warnings =", report["n_warnings"])
for severity in ("error", "warning"):
    shown = [i for i in report["issues"] if i["severity"] == severity][:10]
    for issue in shown:
        print(f"  [{severity}]", issue)

write_m1_tables(paper_rows, reference_rows)
write_json(TABLES_DIR / "validation_report.json", report)
print("\nĐÃ XUẤT:")
print(" - papers.json     ->", TABLES_DIR / "papers.json")
print(" - references.json ->", TABLES_DIR / "references.json")
print(" - validation      ->", TABLES_DIR / "validation_report.json")


In [ ]:
# Kiểm tra nhanh output trên Drive
from pathlib import Path
import json

n_ref_files = len(list(REFS_ROOT.glob("*/references_s2.json")))
print(f"references_s2.json đã lưu: {n_ref_files} paper")

sample = next(REFS_ROOT.glob("*/references_s2.json"), None)
if sample:
    rows = read_jsonl(sample)
    with_arxiv = sum(1 for r in rows if r.get("ref_arxiv_id"))
    print(f"Ví dụ {sample.parent.name}: {len(rows)} refs, {with_arxiv} có arXiv ID")
    print("Một record:", rows[0] if rows else None)

for name in ("papers.json", "references.json", "s2_crawl_records.csv",
             "metrics_s2_references.json", "validation_report.json"):
    path = TABLES_DIR / name
    print(f"- {name}: {path.stat().st_size / 1024:.1f} KiB" if path.exists()
          else f"- {name}: MISSING")

## Sau khi chạy

- Gửi cho người xử lý dữ liệu: `tables/papers.json`, `tables/references.json`, `tables/s2_crawl_records.csv` và thư mục `references/` trên Drive.
- `references/` chứa **mọi** reference (gồm ref không có arXiv ID) — đừng xóa trước khi submit Hugging Face ở M3.
- Chạy đầy đủ cả pipeline (source + S2 + bảng) trên máy:
  `python -m src.m1_crawling.run_m1 --ids assigned_ids.txt`
- Report Jupyter Book đọc `tables/metrics_s2_references.json` + `data/tables/metrics_m1.json` để vẽ figure lúc compile (`book/m1_report.md`).